In [ ]:
from sqlalchemy.testing.suite.test_reflection import metadata

In [ ]:
from qdrant_client.http.models import VectorParams, Distance
from langchain_qdrant import QdrantVectorStore
from langchain_upstage import UpstageEmbeddings, ChatUpstage
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from qdrant_client import QdrantClient
import mysql.connector
import os
import random
from tqdm import tqdm
from dotenv import find_dotenv, load_dotenv
from typing import List
from langchain_core.embeddings import Embeddings
# 1. 환경 변수 로드
load_dotenv(find_dotenv())

# 2. DB 및 Qdrant 설정 (기존 테스트 파일과 동일)
db_config = {
    'host': os.getenv('DB_HOST'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD'),
    'database': os.getenv('DB_NAME')
}

BATCH_SIZE = 400
collection_name = "attractions_overview" 
qdrant_url = os.getenv('QDRANT_URL')

# 3. 모델 초기화

# 질문 생성을 위한 LLM (Upstage Solar 활용)
llm = ChatUpstage(model="solar-mini")

# 4. Qdrant 클라이언트 및 벡터 스토어 (메모리 모드)
client = QdrantClient(":memory:")
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
)


# 1. 래퍼 클래스 정의
class UpstageDualEmbeddings(Embeddings):
    def __init__(self, query_model: Embeddings, doc_model: Embeddings):
        self.query_model = query_model  # 검색(Query)용 모델
        self.doc_model = doc_model      # 문서(Passage) 저장용 모델

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # 문서를 벡터화할 때는 doc_model 사용
        return self.doc_model.embed_documents(texts)

    def embed_query(self, text: str) -> List[float]:
        # 검색 쿼리를 벡터화할 때는 query_model 사용
        return self.query_model.embed_query(text)

# 2. 모델 초기화 (사용자가 작성한 부분)
# 주의: 두 모델의 출력 차원(dimension)은 반드시 같아야 합니다 (예: 둘 다 4096)
query_embedding_model = UpstageEmbeddings(model="embedding-query")   # 예시 모델명
context_embedding_model = UpstageEmbeddings(model="embedding-passage") # 예시 모델명

# 3. 래퍼 클래스로 두 모델 묶기
dual_embedding_model = UpstageDualEmbeddings(
    query_model=query_embedding_model,
    doc_model=context_embedding_model
)

# 4. QdrantVectorStore에 묶은 모델 전달
vector_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=dual_embedding_model, # 여기에 래퍼 객체를 넣습니다.
)


In [13]:
# 5. 데이터 로드 및 인덱싱 (평가를 위한 데이터 준비)
all_docs = []

try:
    connection = mysql.connector.connect(**db_config)
    cursor = connection.cursor(dictionary=True)

    # 테스트를 위해 약 100~200개 정도의 데이터만 로드
    query = f"""
        SELECT *
        FROM attractions
        WHERE overview IS NOT NULL AND overview != ''
        LIMIT 500
    """
    cursor.execute(query)
    rows = cursor.fetchall()

    for row in rows:
        if float(row["latitude"]) != 0 and float(row["longitude"]) != 0:
            doc = Document(
                page_content=row["overview"],
                metadata={
                    "id": row["no"],  # 정확한 평가를 위해 고유 ID 필수
                    "title": row["title"],
                    "location": {
                        "lat": float(row["latitude"]),
                        "lon": float(row["longitude"])
                    }
                }
            )
            all_docs.append(doc)
    
    # Qdrant에 문서 저장
    if all_docs:
        vector_store.add_documents(all_docs)
        print(f"총 {len(all_docs)}개의 문서가 인덱싱되었습니다.")

except Exception as e:
    print(f"Error: {e}")
finally:
    if 'connection' in locals() and connection.is_connected():
        cursor.close()
        connection.close()

총 500개의 문서가 인덱싱되었습니다.


In [ ]:
# 6. 평가 데이터셋 생성 (Synthetic QA Pairs)
# 문서 내용을 바탕으로 LLM이 검색 질문을 생성하도록 함

qa_pairs = []
sample_size = 150  # 평가에 사용할 질문 개수 (전체 문서 중 일부만 샘플링)

print("--- 평가용 질문 생성 시작 ---")

prompt_template = PromptTemplate.from_template(
    """다음은 관광지에 대한 설명입니다:
    
    {context}
    
    위 설명을 바탕으로 사용자가 이 관광지를 찾기 위해 검색할 만한 자연스러운 질문을 한국어로 1개만 만들어주세요.
    질문에는 관광지의 이름이 직접적으로 들어가면 안됩니다.

    사용자는 관광지의 정보를 모른다 가정하고 질문을 입력합니다.
    현재 자신의 상황에서 가고자 하는 관광지를 피상적으로 입력할 것 입니다.

    [예시1]
    근처에 기분 전환 겸 쇼핑하기 좋은 곳 있어? 가급적이면 카페도 있었으면 좋겠고.
    
    [예시2]
    날씨가 추워서 몸을 녹일 수 있는 따뜻한 음료를 즐길 수 있는 카페나 반신욕 할 수 있는 온천 추천해줘.

    질문:"""
)

chain = prompt_template | llm | StrOutputParser()

# 랜덤 샘플링
sampled_docs = random.sample(all_docs, min(sample_size, len(all_docs)))

for doc in tqdm(sampled_docs):
    try:
        # 질문 생성
        question = chain.invoke({"context": doc.page_content[:1000]})
        qa_pairs.append({
            "question": question.strip(),
            "ground_truth_id": doc.metadata["id"], # 정답 문서 ID
            "ground_truth_title": doc.metadata["title"]
        })
    except Exception as e:
        print(f"질문 생성 실패: {e}")

print(f"\n총 {len(qa_pairs)}개의 평가용 질문-정답 쌍 생성 완료")
# 생성된 질문 예시 확인
print(f"예시 Question: {qa_pairs[0]['question']}")
print(f"예시 Target: {qa_pairs[0]['ground_truth_title']}")

--- 평가용 질문 생성 시작 ---


100%|██████████| 150/150 [01:22<00:00,  1.82it/s]


총 150개의 평가용 질문-정답 쌍 생성 완료
예시 Question: 성북동에 있는 독특한 성모상과 아름다운 건축물로 유명한, 사진 찍기 좋은 장소 어디인가요?
예시 Target: 성북동성당


In [18]:
print(f"예시 Question: {qa_pairs[3]['question']}")
print(f"예시 Target: {qa_pairs[3]['ground_truth_title']}")

예시 Question: 종로나 경복궁 근처에서 한옥의 아름다움과 전통적인 골목길을 즐길 수 있는 곳이 있을까요? 또한 그 주변에 오래된 맛집이나 미술관, 도서관 같은 다른 볼거리가 많은 곳이 어디인지 궁금합니다.
예시 Target: 북촌 8경


In [ ]:
import math
from tqdm import tqdm

# 7. 검색 성능 평가 (Hit Rate, MRR, NDCG 측정)

def calculate_metrics(retriever, qa_pairs, k=5):
    hits = 0
    mrr_sum = 0
    ndcg_sum = 0  # NDCG 합계 변수 초기화

    print(f"--- Top-{k} 검색 평가 시작 ---")
    
    for item in tqdm(qa_pairs):
        query = item["question"]
        target_id = item["ground_truth_id"]
        
        # 검색 수행
        results = retriever.similarity_search_with_score(query, k=k)
        top_k_scores = [score for _, score in results]

        #[(Document, score), (Document, score), ...]
        # 정답 확인
        found = False
        for rank, doc in enumerate([doc for doc, _ in results]):
            # 메타데이터의 ID로 비교
            if doc.metadata.get("id") == target_id:
                # 1. Hit Rate 계산용
                hits += 1
                
                # 2. MRR 계산용 (1 / 순위)
                mrr_sum += 1.0 / (rank + 1)
                
                # 3. NDCG 계산용
                # 공식: DCG = rel_i / log2(i + 1)
                # 여기서는 0-index이므로 rank+1이 실제 순위 i가 됩니다.
                # 따라서 분모는 log2((rank + 1) + 1) = log2(rank + 2)가 됩니다.
                # 정답이 1개인 경우 IDCG는 1이므로, DCG가 곧 NDCG가 됩니다.
                ndcg_sum += 1.0 / math.log2(rank + 2)
                
                found = True
                break
        
        if not found:
            pass # 디버깅용: print(f"Missed: {query}")

    # 평균 점수 계산
    n = len(qa_pairs)
    hit_rate = hits / n
    mrr = mrr_sum / n
    ndcg = ndcg_sum / n  # 평균 NDCG
    
    return hit_rate, mrr, ndcg, top_k_scores


--- Top-5 검색 평가 시작 ---


100%|██████████| 150/150 [01:33<00:00,  1.60it/s]


ValueError: too many values to unpack (expected 3)

In [24]:
hit_rate, mrr, ndcg, scores = calculate_metrics(vector_store, qa_pairs, k=5)

print("\n=== 평가 결과 (Dense Search) ===")
print(f"Hit Rate @ 5: {hit_rate:.4f}")
print(f"MRR @ 5:      {mrr:.4f}")
print(f"NDCG @ 5:     {ndcg:.4f}")
print(f"Top-5 Score:  {scores}")
print("===============================")

--- Top-5 검색 평가 시작 ---


100%|██████████| 150/150 [01:41<00:00,  1.47it/s]


=== 평가 결과 (Dense Search) ===
Hit Rate @ 5: 0.8533
MRR @ 5:      0.7440
NDCG @ 5:     0.7713
Top-5 Score:  [0.5561888845896029, 0.43965338617317695, 0.4170583310837357, 0.4169962195131995, 0.4071046947289886]
